# Analysis of Woody Cover and Yearly FireCCI Combined

Version: 1.0 (2025-06-06) © R Butzer

This notebook allows you to interactively analyze the wildE woody cover tiles based on inpujt Coordinates as well as the annual FireCCI Burned Area from 2000 to 2020 for an input feature.  

### How to Use:
1. Chose woody cover tiles by inputing X and Y Coordinates
2. Draw a polygon or rectangle on the map to select a region of interest (preferably the woody tile but can be anything)
2. Use the sliders to select woody cover years (1985 to 2024) and fire years (2001-2020)
3. The notebook will display the burned area for the selected year within the drawn region as a colored map layer.
4. To choose a new fire area, simply delete the current feature by clicking the "Delete" button


### Points of Improvement:
- Add a button to clear the map and reset the selection.
- Implement a feature to download the selected burned area data as a GeoJSON or CSV file.
- chose Countries from dropdown menu

In [1]:
import os, re, time
import ee
import geemap
import folium
from osgeo import gdal, ogr, osr
gdal.UseExceptions()
import ipywidgets as widgets

# Earth Engine initialisieren
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

In [2]:
def tile_filename(x, y):
    return f"tileID-X{x:03d}-Y{y:03d}_WoodyCover_1985-2024.tif"


# Bereich für X und Y angeben (inklusive Endwert)

# safe x and y value to variables 

Spain = {
    'x_start': 20,
    'x_end': 26,
    'y_start': 26,
    'y_end': 33
}

Germany = {
    'x_start': 5,
    'x_end': 15,
    'y_start': 47,
    'y_end': 55
}

TileEU_75_X040_Y028 = {
    'x_start': 40,
    'x_end': 40,
    'y_start': 28,
    'y_end': 28
}

TileEU_150km_X021_Y027 = {
    'x_start': 21,
    'x_end': 21,
    'y_start': 27,
    'y_end': 27
}


# Wähle den gewünschten Ort:
region = TileEU_150km_X021_Y027  

# Werte automatisch übernehmen
x_start = region['x_start']
x_end = region['x_end']
y_start = region['y_start']
y_end = region['y_end']






In [ ]:
# Wähle den gewünschten Ort:
region = TileEU_150km_X021_Y027  

# Werte automatisch übernehmen
x_start = region['x_start']
x_end = region['x_end']
y_start = region['y_start']
y_end = region['y_end']

In [ ]:
def tile_filename(x, y):
    return f"tile_X{x:03d}_Y{y:03d}.tif"

x_list = list(range(x_start, x_end + 1))
y_list = list(range(y_start, y_end + 1))

raster_folder = r"A:\_BioGeo\wildE\_PREDICTIONS\Run03_April2025\02_WoodyCover"
vrt_cache = r"A:\_BioGeo\wildE\_PREDICTIONS\Run03_April2025\vrt_cache"
os.makedirs(vrt_cache, exist_ok=True)
vrt_name = f"vrt_X{x_start:03d}-X{x_end:03d}_Y{y_start:03d}-Y{y_end:03d}.vrt"
vrt_path = os.path.join(vrt_cache, vrt_name)

tile_names = [tile_filename(x, y) for x in x_list for y in y_list]
tile_paths = [os.path.join(raster_folder, name) for name in tile_names]

if not os.path.exists(vrt_path):
    gdal.BuildVRT(vrt_path, tile_paths)
    print(f"VRT wurde neu erstellt: {vrt_path}")
else:
    print(f"VRT bereits vorhanden: {vrt_path}")

# --- Map-Setup ---
Map = geemap.Map(basemap='SATELLITE')
Map.center = [54.5260, 15.2551]
Map.zoom = 4

# --- Woody Cover Slider ---
woody_years = list(range(1985, 2025))
woody_slider = widgets.SelectionSlider(
    options=[(str(year), band) for band, year in enumerate(woody_years, start=1)],
    value=1,
    description='Woody Jahr',
    continuous_update=False
)

def update_woody(change):
    # Nur Basemap und Fire-Layer behalten
    Map.layers = [Map.layers[0]] + [l for l in Map.layers if 'FireCCI' in l.name]
    Map.add_raster(
        vrt_path,
        bands=[woody_slider.value],
        layer_name=f"WoodyCover {woody_years[woody_slider.value-1]}",
        vmin=0,
        vmax=100,
        palette="Greens"
    )

woody_slider.observe(update_woody, names='value')
update_woody(None)

# --- FireCCI Setup ---
fire_years = list(range(2001, 2021))
baVis = {
    'min': 1,
    'max': 366,
    'palette': [
        'ff0000', 'fd4100', 'fb8200', 'f9c400', 'f2ff00', 'b6ff05',
        '7aff0a', '3eff0f', '02ff15', '00ff55', '00ff99', '00ffdd',
        '00ddff', '0098ff', '0052ff', '0210ff', '3a0dfb', '7209f6',
        'a905f1', 'e102ed', 'ff00cc', 'ff0089', 'ff0047', 'ff0004'
    ]
}
fire_slider = widgets.SelectionSlider(
    options=fire_years,
    value=fire_years[0],
    description='Fire Jahr',
    continuous_update=False
)
burned_area_label = widgets.Label(value="Burned Area: ... ha")
current_geom = [None]

def update_fire(change):
    # Nur Basemap und Woody-Layer behalten
    Map.layers = [Map.layers[0]] + [l for l in Map.layers if 'WoodyCover' in l.name]
    geom = current_geom[0]
    if geom is None:
        burned_area_label.value = "Bitte ein Polygon zeichnen."
        return
    year = fire_slider.value
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    fire = ee.ImageCollection('ESA/CCI/FireCCI/5_1') \
        .filterDate(start, end) \
        .select('BurnDate')
    if fire.size().getInfo() > 0:
        max_burn = fire.max().clip(geom)
        Map.addLayer(max_burn, baVis, f'FireCCI {year}')
        burned = fire.map(lambda img: img.gt(0)).sum()
        pixel_area = burned.multiply(ee.Image.pixelArea())
        stats = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geom,
            scale=250,
            maxPixels=1e10
        )
        area_m2 = stats.get('BurnDate').getInfo()
        if area_m2 is not None:
            area_ha = area_m2 / 10000
            burned_area_label.value = f"Burned Area: {area_ha:,.1f} ha"
        else:
            burned_area_label.value = "Burned Area: 0 ha"
    else:
        burned_area_label.value = "Burned Area: 0 ha"

def handle_draw(target, action, geo_json):
    geom = geemap.geojson_to_ee(geo_json)
    current_geom[0] = geom
    update_fire(None)

draw_control = Map.draw_control
draw_control.polygon = {
    "shapeOptions": {"color": "#FF0000", "weight": 3, "fillOpacity": 0}
}
draw_control.rectangle = {
    "shapeOptions": {"color": "#FF0000", "weight": 3, "fillOpacity": 0}
}
draw_control.on_draw(handle_draw)
fire_slider.observe(update_fire, names='value')

# --- UI ---
ui = widgets.VBox([
    widgets.HBox([woody_slider, fire_slider, burned_area_label]),
    Map
])
display(ui)

In [3]:
import os
import ipywidgets as widgets
import geemap
import ee
from osgeo import gdal


def tile_filename(x, y):
    return f"tile_X{x:03d}_Y{y:03d}.tif"

x_list = list(range(x_start, x_end + 1))
y_list = list(range(y_start, y_end + 1))

raster_folder = r"A:\_BioGeo\wildE\_PREDICTIONS\Run03_April2025\02_WoodyCover"
vrt_cache = r"A:\_BioGeo\wildE\_PREDICTIONS\Run03_April2025\vrt_cache"
os.makedirs(vrt_cache, exist_ok=True)
vrt_name = f"vrt_X{x_start:03d}-X{x_end:03d}_Y{y_start:03d}-Y{y_end:03d}.vrt"
vrt_path = os.path.join(vrt_cache, vrt_name)

tile_names = [tile_filename(x, y) for x in x_list for y in y_list]
tile_paths = [os.path.join(raster_folder, name) for name in tile_names]

if not os.path.exists(vrt_path):
    gdal.BuildVRT(vrt_path, tile_paths)
    print(f"VRT wurde neu erstellt: {vrt_path}")
else:
    print(f"VRT bereits vorhanden: {vrt_path}")

# --- Map-Setup ---
Map = geemap.Map(basemap='SATELLITE')
Map.center = [54.5260, 15.2551]
Map.zoom = 4

# --- Woody Cover Slider ---
woody_years = list(range(1985, 2025))
woody_slider = widgets.SelectionSlider(
    options=[(str(year), band) for band, year in enumerate(woody_years, start=1)],
    value=1,
    description='Woody Jahr',
    continuous_update=False
)

def update_woody(change):
    # Nur Basemap und Fire-Layer behalten
    Map.layers = [Map.layers[0]] + [l for l in Map.layers if 'FireCCI' in l.name]
    Map.add_raster(
        vrt_path,
        bands=[woody_slider.value],
        layer_name=f"WoodyCover {woody_years[woody_slider.value-1]}",
        vmin=0,
        vmax=100,
        palette="Greens"
    )

woody_slider.observe(update_woody, names='value')
update_woody(None)

# --- FireCCI Setup ---
fire_years = list(range(2000, 2021))
baVis = {
    'min': 1,
    'max': 366,
    'palette': [
        'ff0000', 'fd4100', 'fb8200', 'f9c400', 'f2ff00', 'b6ff05',
        '7aff0a', '3eff0f', '02ff15', '00ff55', '00ff99', '00ffdd',
        '00ddff', '0098ff', '0052ff', '0210ff', '3a0dfb', '7209f6',
        'a905f1', 'e102ed', 'ff00cc', 'ff0089', 'ff0047', 'ff0004'
    ]
}
fire_slider = widgets.SelectionSlider(
    options=fire_years,
    value=fire_years[0],
    description='Fire Jahr',
    continuous_update=False
)
burned_area_label = widgets.Label(value="Burned Area: ... ha")
current_geom = [None]

def update_fire(change):
    # Nur Basemap und Woody-Layer behalten
    Map.layers = [Map.layers[0]] + [l for l in Map.layers if 'WoodyCover' in l.name]
    geom = current_geom[0]
    if geom is None:
        burned_area_label.value = "Bitte ein Polygon zeichnen."
        return
    year = fire_slider.value
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    fire = ee.ImageCollection('ESA/CCI/FireCCI/5_1') \
        .filterDate(start, end) \
        .select('BurnDate')
    if fire.size().getInfo() > 0:
        max_burn = fire.max().clip(geom)
        Map.addLayer(max_burn, baVis, f'FireCCI {year}')
        burned = fire.map(lambda img: img.gt(0)).sum()
        pixel_area = burned.multiply(ee.Image.pixelArea())
        stats = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geom,
            scale=250,
            maxPixels=1e10
        )
        area_m2 = stats.get('BurnDate').getInfo()
        if area_m2 is not None:
            area_ha = area_m2 / 10000
            burned_area_label.value = f"Burned Area: {area_ha:,.1f} ha"
        else:
            burned_area_label.value = "Burned Area: 0 ha"
    else:
        burned_area_label.value = "Burned Area: 0 ha"

def handle_draw(target, action, geo_json):
    geom = geemap.geojson_to_ee(geo_json)
    current_geom[0] = geom
    update_fire(None)

draw_control = Map.draw_control
draw_control.polygon = {
    "shapeOptions": {"color": "#FF0000", "weight": 3, "fillOpacity": 0}
}
draw_control.rectangle = {
    "shapeOptions": {"color": "#FF0000", "weight": 3, "fillOpacity": 0}
}
draw_control.on_draw(handle_draw)
fire_slider.observe(update_fire, names='value')

# --- UI ---
ui = widgets.VBox([
    widgets.HBox([woody_slider, fire_slider, burned_area_label]),
    Map
])
display(ui)

VRT bereits vorhanden: A:\_BioGeo\wildE\_PREDICTIONS\Run03_April2025\vrt_cache\vrt_X021-X021_Y027-Y027.vrt
